# 06 BM25 Baseline Evaluation

## 목적
BM25 retrieval baseline을 고정 평가셋으로 측정한다.

이 notebook은 구현 코드가 아니라 실행 결과 확인용이다. 핵심 구현은 `app.infrastructure.index.bm25`와 `pipelines.run_bm25_baseline_eval`에 둔다.


## 입력과 공개 정책

- 평가셋: `../evals/datasets/retrieval_eval_seed.jsonl`
- 결과: `../evals/results/bm25_baseline_results.jsonl`
- 리포트: `../evals/reports/bm25_baseline_report.md`
- private chunk: `<private parent_child_chunks report>`

public repository에는 원문 PDF, 전체 parser JSON, 전체 chunk text를 포함하지 않는다.


In [ ]:
from pathlib import Path

from app.domain.retrieval import (
    collect_retrieval_eval_dataset_failures,
    load_retrieval_eval_jsonl,
    summarize_retrieval_eval_dataset,
)

dataset_path = Path("../evals/datasets/retrieval_eval_seed.jsonl")
items = load_retrieval_eval_jsonl(dataset_path)
summary = summarize_retrieval_eval_dataset(items)
failures = collect_retrieval_eval_dataset_failures(summary)

summary.model_dump(), failures


## v2 Split Contract

평가셋 v2는 `metadata.split`, `difficulty`, `place_ids`, `requires_context`, `answerability`, `review_status`를 사용한다. 현재 `seed` split은 smoke test이며 contract gate만 통과한다. split readiness gate는 query type별 `dev 10`, `test 5` 목표를 채우기 전까지 실패로 기록한다.


In [ ]:
{
    "dataset_version": summary.dataset_version,
    "split_distribution": summary.split_distribution,
    "query_type_by_split": summary.query_type_by_split,
    "dev_target_shortfall_count": summary.dev_target_shortfall_count,
    "test_target_shortfall_count": summary.test_target_shortfall_count,
    "duplicate_query_id_count": summary.duplicate_query_id_count,
    "missing_metadata_count": summary.missing_metadata_count,
    "answerability_mismatch_count": summary.answerability_mismatch_count,
    "voice_followup_context_missing_count": summary.voice_followup_context_missing_count,
}


In [ ]:
import json

results_path = Path("../evals/results/bm25_baseline_results.jsonl")
rows = [json.loads(line) for line in results_path.read_text(encoding="utf-8").splitlines()]

{
    "result_row_count": len(rows),
    "query_count": len({row["query_id"] for row in rows}),
    "query_types": sorted({row["query_type"] for row in rows}),
    "max_rank": max(row["rank"] for row in rows if row["rank"] is not None),
}


In [ ]:
report_path = Path("../evals/reports/bm25_baseline_report.md")
report = report_path.read_text(encoding="utf-8")

required_fragments = [
    "Recall@5 | 0.250000",
    "MRR | 0.152778",
    "nDCG@5 | 0.120124",
    "public_raw_text_leakage_count | 0",
    "private_path_leakage_count | 0",
]

[fragment for fragment in required_fragments if fragment not in report]


## 정량 결과

- `Recall@1=0.083333`
- `Recall@3=0.250000`
- `Recall@5=0.250000`
- `MRR=0.152778`
- `nDCG@5=0.120124`
- `latency_p95_ms`: 실행 환경에 따라 변동되므로 report 값을 기준으로 확인한다.
- `missing_result_count=0`
- `abstain_with_candidate_count=2`

이 결과는 성능 개선 주장이 아니라 Dense/Hybrid/query rewrite 비교를 위한 기준선이다.


## 정성 판단

- `place_story`, `relationship`, `route_context` 일부는 lexical match만으로도 top-5에 도달했다.
- `place_fact`, `overview`, `voice_followup`은 seed 기준 top-5 recall이 0이다.
- `no_answer`는 후보를 반환했다. BM25는 검색기라 corpus 밖 질문을 독립적으로 거절하지 못한다.
- 다음 단계에서 Dense/Hybrid retrieval과 query rewrite가 어느 query type을 개선하는지 비교해야 한다.
